### We're just going to load in some data, visualize it and make some figures, and compute a few different things just to get an understanding of what our larvae are doing

The most important step in doing any kind of data analysis is understanding the dataset you're working with. It is possible with some lines of code to run analyses in increasing levels of sophistication, but if you don't have an intuitive sense of what your data looks like you can very easily be led astray by a bunch of numbers. 

This should always be your first step with any new dataset you're working with. It's sometimes referred to as 'exploratory data analysis'.

# Step **0**: Find your data
I'm just going to copy here some paths to different sets of data we've (all you, really!) collected so we can look at them. Any set of data folders will do just fine! I'm loading in some of the gaussian condition data here:

In [ ]:
gaussian1 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/average_plume'
gaussian2 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot'

# Step **1**: Load in the data
We're gonna use pandas to read in the csv files as pandas DataFrames (you could also use numpy if you prefer). We're also going to need to write a function that will find the files we want. Here's a simple one you could use (after importing some packages):

In [ ]:
import os
import pandas as pd
from glob import glob

In [ ]:
def get_csv_files(dir_path, suffix=''):
    """
    Get all the csv files in the given directory recursively. This means it'll look in all subdirectories as well.
    """
    return glob(os.path.join(dir_path, '**', f'*{suffix}.csv')) 

I added the suffix argument so you can specify which csv files you want. For raw data, this'll just be `'data'`. Another thing you might want to do is have a look at the other files that appear in your data folder (other than the data csv), so let's go ahead and do that.

First, I need to get the name of each directory that has a data file I want. This is easy with `os.path.dirname()`:

In [ ]:
csv_files = get_csv_files(gaussian1, suffix='level1') # I'm just looking at the gaussian1 directory for now
data_dirs = [os.path.dirname(f) for f in csv_files]

So now we have a list of paths to the data directories that contain the csv files I wanted. Each of these directories should contain the following files:
- experiment_settings.json
- sm_raw.npy
- sm_thresh.npy
- sm_skeletons.npy
- Background.tiff
- The stimulus .csv file, this won't necessarily have a standard name and will just named similar to the VR arena csv file we use

We'll ignore other files for now. 

Are all these very important to look at? not really. 

But we need to be aware of the full set of data we have, and that includes all our files, not just the data .csv

Luckily, we can read them in pretty easily:

In [ ]:
sm_raw_files = [os.path.join(d, 'sm_raw.npy') for d in data_dirs]

.npy files are just a way to store a numpy array, so let's look at one:

In [ ]:
import numpy as np

sm_raw0 = sm_raw_files[0]
raw = np.load(sm_raw0)

print(raw.shape) #what does the data look like?

so it's a 3D array, looks like there's a n x m matrix for each frame in our data. Let's visualize the first frame:

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(raw[:, :, 0], cmap='gray')

That's the image of our larva! This is what the `start_GUI.py` is loading when we do the manual swap correction. See what it looks like at different frames too, and also have a look at what the `sm_thresh.npy` and `sm_skeleton.npy` files contain too.

 Just going to casially leave an empty code block for you here:

In [ ]:
sm_skeletons_files = [os.path.join(d, 'sm_skeletons.npy') for d in data_dirs]
sm_skeletons0 = sm_skeletons_files[0]
skeletons = np.load(sm_skeletons0)
print(skeletons.shape) #what does the data look like?
plt.imshow(skeletons[:, :, 2000], cmap='gray')
plt.show()

sm_thresh_files = [os.path.join(d, 'sm_thresh.npy') for d in data_dirs]
sm_thresh0 = sm_thresh_files[0]
thresh = np.load(sm_thresh0)
print(thresh.shape) #what does the data look like?
plt.imshow(thresh[:, :, 2000], cmap='gray')

The other file we want to pay attention to is the `experiment_settings.json`, this contains all the metadata that PiVR saves, so it's contains some useful information. A JSON file is basically a text file with a specific structure. Loading these is easy too:

In [ ]:
import json

exp_settings_files = [os.path.join(d, 'experiment_settings.json') for d in data_dirs]

exp_set0 = exp_settings_files[0]
with open(exp_set0, 'r') as f:
    exp_settings = json.load(f)

When you load in a JSON file you get a dictionary with all the keys and values that are in the JSON, so let's have a look:

In [ ]:
print(exp_settings.keys()) # what are the keys in the json file?

for key in exp_settings.keys():
    print(key + " : " + str(exp_settings[key])) # what are the values of the keys?    

Particularly useful keys you'll want to make use out of are `'Framerate'`, `'Source x'`, `'Source y'`, and `'Pixel per mm'`. There might also be a `experiment_settings_updated.json` in your data folders. Those are duplicates of `experiment_settings.json` with some values changed/added. I do this because it's never a good idea to overwrite your raw data, so I keep the file I modify separate. But feel free to load that and look at it too! `'Pixel per mm'` will be different.

Okay we're actually going to read in some data now, a single raw data file in this case: 

In [ ]:
data_file = csv_files[0]
data = pd.read_csv(data_file)
print(data_file)

print(data.head()) # what does the data look like?
print(data.columns) # what are the columns in the data?


The only columns we're concerned about for now are `'X-Centroid'`, `'Y-Centroid'`, `'X-Head'`,
       `'Y-Head'`, `'X-Tail'`, `'Y-Tail'`, `'X-Midpoint'`, `'Y-Midpoint'`. We can access them like this:

In [ ]:
data['X-Centroid'] # this will return a dataframe with the X-Centroid column
data['X-Centroid'].values # this will return a numpy array with the X-Centroid column


so for example I can plot the whole trajectory path:

In [ ]:
plt.plot(data['X-Centroid'].values, data['Y-Centroid'].values)

The first or last few data points often cause issues, so we can just slice off the first and last seconds of the data:

In [ ]:
plt.plot(data['X-Centroid'].values[30:-30], data['Y-Centroid'].values[30:-30])

if you zoom in (you can use `plt.xlim()` and `plt.ylim()` to specify the limits of your plot axes) you'll notice that the path is quite pixelated. Try computing the speed of the larva (`np.gradient()` is useful for this) and you'll see that it's a genuine problem.

After you do that, try filtering the data first and then computing the speed. 

You can filter with the gaussian filter function from scipy: (import using `from scipy.ndimage import gaussian_filter`), or the Savitzky-Golay filter (import using `from scipy.signal import savgol_filter`).

You'll need to specify some parameters, try different values and look at your trajectory and speed - try to get a sense of what the filter is doing and what the best parameters might be. 

In [ ]:
import numpy as np
from math import sqrt
from scipy.ndimage import gaussian_filter
from scipy.signal import savgol_filter
plt.xlim(790, 840)
plt.ylim(790, 870)
plt.plot(data['X-Centroid'].values[30:-30], data['Y-Centroid'].values[30:-30])

 
x = data['X-Centroid'].values[300:-30]
y = data['Y-Centroid'].values[300:-30]
x_filt_gaus = gaussian_filter(x, sigma=5)
y_filt_gaus = gaussian_filter(y, sigma=5)
x_filt = savgol_filter(x, 501, 3)
y_filt = savgol_filter(y, 501, 3)
plt.plot(x_filt, y_filt)
#plt.show()
# plt.plot(np.gradient(x))
# plt.plot(np.gradient(x_filt))
#plt.plot(np.gradient(data['Y-Centroid'].values[30:-30]))

# plt.plot(sqrt(np.gradient(data['X-Centroid'].values[30:-30])**2 + np.gradient(data['Y-Centroid'].values[30:-30])**2))
# plt.plot(np.gradient(data['X-Centroid'].values[30:-30], 5), np.gradient(data['Y-Centroid'].values[30:-30], 5))
# plt.show()
# plt.plot(np.gradient(y))
# plt.plot(np.gradient(y_filt))



In [ ]:
# Calculate the speed based on unfiltered positions
speed = np.sqrt(np.gradient(x)**2 + np.gradient(y)**2)

# Plot the speed
plt.plot(speed)
plt.xlabel('Frame')
plt.ylabel('Speed (pixels/frame)')
plt.title('Speed based on Unfiltered Positions')
plt.xlim(0, 800)
plt.show()

In [ ]:
# Calculate the speed based on filtered positions
speed = np.sqrt(np.gradient(x_filt_gaus)**2 + np.gradient(y_filt_gaus)**2)

# Plot the speed
plt.plot(speed)
plt.xlabel('Frame')
plt.ylabel('Speed (pixels/frame)')
plt.title('Speed based on (Gaussian) Filtered Positions')
plt.xlim(0, 800)
plt.ylim(0, 0.5)
plt.show()

In [ ]:
# Calculate the speed based on filtered positions
speed = np.sqrt(np.gradient(x_filt)**2 + np.gradient(y_filt)**2)

# Plot the speed
plt.plot(speed)
plt.xlabel('Frame')
plt.ylabel('Speed (pixels/frame)')
plt.title('Speed based on (Savgol) Filtered Positions')
plt.xlim(0,7000)
plt.ylim(0, 1)
# plt.xlim(0, 800)
plt.show()

In [ ]:
plt.scatter(data['stimulation'].values[300:-30], speed, label='Filtered Speed')
plt.xlabel('Stimulation')
plt.ylabel('Speed (pixels/frame)')
plt.title('Speed vs Stimulation')
plt.legend()
plt.show()

In [ ]:
stim = data['stimulation'].values[300:-30]
# plt.hist(stim, bins=20)
stim1000 = np.where(stim <= 800)[0]
stim8000 = np.where(stim >= 8000)[0]

plt.boxplot([speed[stim1000], speed[stim8000]], labels=['Stimulus 1000', 'Stimulus 8000'], showfliers=False)

print(len(stim1000))
print(len(stim8000))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# Initialize lists to store speeds and stimulations
all_speeds = []
all_stimulations = []

# Loop through all CSV files
for file in csv_files:
    # Read the data
    data = pd.read_csv(file)
    
    # Extract X and Y Centroid values
    x = data['X-Centroid'].values[30:-30]
    y = data['Y-Centroid'].values[30:-30]
    
    # Apply Savitzky-Golay filter to smooth the data
    x_filt = savgol_filter(x, window_length=5, polyorder=3)
    y_filt = savgol_filter(y, window_length=5, polyorder=3)
    
    # Calculate speed
    speed = np.sqrt(np.gradient(x_filt)**2 + np.gradient(y_filt)**2)
    
    # Smooth the speed data using Savitzky-Golay filter
    smoothed_speed = savgol_filter(speed, window_length=31, polyorder=1)
    
    # Extract stimulation values
    stim = data['stimulation'].values[30:-30]
    
    # Append to the lists
    all_speeds.extend(smoothed_speed)
    all_stimulations.extend(stim)

# Convert to numpy arrays for easier processing
all_speeds = np.array(all_speeds)
all_stimulations = np.array(all_stimulations)

# Group by stimulation levels and calculate average speed
unique_stimulations = np.unique(all_stimulations)
average_speeds = [np.mean(all_speeds[all_stimulations == stim]) for stim in unique_stimulations]

# Fit a line of best fit (linear regression)
coefficients = np.polyfit(unique_stimulations, average_speeds, 1)  # Linear fit
line_of_best_fit = np.polyval(coefficients, unique_stimulations)

# Plot the average speed vs stimulation graph
plt.plot(unique_stimulations, average_speeds, marker='.', label='Average Speed')
plt.plot(unique_stimulations, line_of_best_fit, color='red', linestyle='--', label='Line of Best Fit')
plt.xlabel('Stimulation')
plt.ylabel('Average Speed (pixels/frame)')
plt.title('Average Speed vs Stimulation (Smoothed with Savitzky-Golay)')
#plt.xlim(2130, 2150)
#plt.ylim(-0.2, 1)
plt.legend()
plt.grid()
plt.show()
print("Unique Stimulations:", unique_stimulations)
print("Average Speeds:", average_speeds)
print("Line of Best Fit:", line_of_best_fit)

Calculating length of larvae

In [ ]:
# # Specify the frame index you want to inspect
# frame_index = 6500  # Replace with the desired frame index

# # Extract the head and tail positions for the specified frame
# x_head = data.loc[frame_index, 'X-Head']
# y_head = data.loc[frame_index, 'Y-Head']
# x_tail = data.loc[frame_index, 'X-Tail']
# y_tail = data.loc[frame_index, 'Y-Tail']

# # Print the positions
# print(f"Frame {frame_index}:")
# print(f"Head Position: (X: {x_head}, Y: {y_head})")
# print(f"Tail Position: (X: {x_tail}, Y: {y_tail})")

# # Print the length of the larva
# length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)
# print(f"Length of Larva: ~{length:.2f} pixels")
# ppmm = exp_settings['Pixel per mm']
# print(f"Length of Larva: ~{(length/ppmm):.2f} mm")
'''
iterations = 0
added_lengths_PIX = 0
added_lengths_MM = 0

for frame_index in range(0, len(data), 50):
    if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
        continue
    # Extract the head and tail positions for the specified frame
    x_head = data.loc[frame_index, 'X-Head']
    y_head = data.loc[frame_index, 'Y-Head']
    x_tail = data.loc[frame_index, 'X-Tail']
    y_tail = data.loc[frame_index, 'Y-Tail']

    # Print the length of the larva
    length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)
    ppmm = exp_settings['Pixel per mm']
    length_mm = length / ppmm 
    added_lengths_PIX += length
    added_lengths_MM += length_mm
    iterations += 1
print(f"Average Length of Larva: ~{(added_lengths_PIX/iterations):.2f} pixels")
print(f"Average Length of Larva: ~{(added_lengths_MM/iterations):.2f} mm")
'''

def calculate_average_length(data, exp_settings, step=50):
    """
    Calculate the average length of the larva in millimeters.

    Parameters:
        data (pd.DataFrame): The DataFrame containing the larva data.
        exp_settings (dict): A dictionary containing experiment settings, including 'Pixel per mm'.
        step (int): The step size for iterating through frames (default is 50).

    Returns:
        float: The average length of the larva in millimeters.
    """
    iterations = 0
    added_lengths_MM = 0

    # Ensure 'Pixel per mm' is valid
    if 'Pixel per mm' not in exp_settings or exp_settings['Pixel per mm'] <= 0:
        raise ValueError("Invalid 'Pixel per mm' value in exp_settings.")

    ppmm = exp_settings['Pixel per mm']

    # Loop through frames with the specified step
    for frame_index in range(0, len(data), step):
        # Check for missing data
        if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
            continue

        # Extract the head and tail positions for the specified frame
        x_head = data.loc[frame_index, 'X-Head']
        y_head = data.loc[frame_index, 'Y-Head']
        x_tail = data.loc[frame_index, 'X-Tail']
        y_tail = data.loc[frame_index, 'Y-Tail']

        # Calculate the length of the larva in pixels
        length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)

        # Convert length to millimeters
        length_mm = length / ppmm

        # Accumulate lengths
        added_lengths_MM += length_mm
        iterations += 1

    # Calculate and return the average length in millimeters
    if iterations > 0:
        return added_lengths_MM / iterations
    else:
        raise ValueError("No valid frames to calculate the average length.")


In [ ]:
def calculate_average_of_75th_percentile(data, exp_settings, step=50):
    """
    Calculate the average of all larva lengths in the 75th percentile in millimeters.

    Parameters:
        data (pd.DataFrame): The DataFrame containing the larva data.
        exp_settings (dict): A dictionary containing experiment settings, including 'Pixel per mm'.
        step (int): The step size for iterating through frames (default is 50).

    Returns:
        float: The average of all larva lengths in the 75th percentile in millimeters.
    """
    lengths_mm = []

    # Ensure 'Pixel per mm' is valid
    if 'Pixel per mm' not in exp_settings or exp_settings['Pixel per mm'] <= 0:
        raise ValueError("Invalid 'Pixel per mm' value in exp_settings.")

    ppmm = exp_settings['Pixel per mm']

    # Loop through frames with the specified step
    for frame_index in range(0, len(data), step):
        # Check for missing data
        if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
            continue

        # Extract the head and tail positions for the specified frame
        x_head = data.loc[frame_index, 'X-Head']
        y_head = data.loc[frame_index, 'Y-Head']
        x_tail = data.loc[frame_index, 'X-Tail']
        y_tail = data.loc[frame_index, 'Y-Tail']

        # Calculate the length of the larva in pixels
        length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)

        # Convert length to millimeters and store it
        length_mm = length / ppmm
        lengths_mm.append(length_mm)

    # Calculate the 75th percentile threshold
    if lengths_mm:
        percentile_75_threshold = np.percentile(lengths_mm, 75)

        # Filter lengths that are in the 75th percentile
        lengths_in_75th_percentile = [length for length in lengths_mm if length >= percentile_75_threshold]

        # Calculate and return the average of the 75th percentile lengths
        if lengths_in_75th_percentile:
            return np.mean(lengths_in_75th_percentile)
        else:
            raise ValueError("No lengths found in the 75th percentile.")
    else:
        raise ValueError("No valid frames to calculate the 75th percentile.")

In [ ]:
average_length_mm = calculate_average_length(data, exp_settings, step=50)
print(f"Average Length of Larva in one experiment: ~{average_length_mm:.2f} mm")
file_count = 0
# Loop through all CSV files in the folder
for file in csv_files:
    # Read the data from the current file
    data = pd.read_csv(file)
    file_count += 1
    # Print the file name for reference
    print(f"Processing file: file {file_count}")
    
    # Call the function to calculate the average length
    try:
        average_length_mm = calculate_average_length(data, exp_settings, step=50)
        print(f"Average Length of Larva in file {file_count}: ~ {average_length_mm:.2f} mm")
    except ValueError as e:
        print(f"Error processing file file {file_count}: {e}")

# Calculate the average of all larva lengths in the 75th percentile
for file in csv_files:
    # Read the data from the current file
    data = pd.read_csv(file)
    file_count += 1
    # Print the file name for reference
    print(f"Processing file: file {file_count}")
    
    # Call the function to calculate the average length
    try:
        average_length_mm = calculate_average_of_75th_percentile(data, exp_settings, step=50)
        print(f"75th percentile Average Length of Larva in file {file_count}: ~ {average_length_mm:.2f} mm")
    except ValueError as e:
        print(f"Error processing file file {file_count}: {e}")

In [ ]:
import matplotlib.pyplot as plt

# Initialize a list to store the 75th percentile average lengths
percentile_75_lengths = []

# Loop through all CSV files in the folder
for file_index, file in enumerate(csv_files, start=1):
    # Read the data from the current file
    data = pd.read_csv(file)
    
    # Print the file name for reference
    # print(f"Processing file {file_index}: {file}")
    
    # Call the function to calculate the 75th percentile average length
    try:
        average_length_mm = calculate_average_of_75th_percentile(data, exp_settings, step=50)
        percentile_75_lengths.append(average_length_mm)
        print(f"75th Percentile Average Length of Larva in file {file_index}: ~{average_length_mm:.2f} mm")
    except ValueError as e:
        print(f"Error processing file {file_index}: {e}")
        percentile_75_lengths.append(None)  # Append None for files with errors

# Filter out None values for plotting
valid_lengths = [length for length in percentile_75_lengths if length is not None]
valid_file_indices = [i + 1 for i, length in enumerate(percentile_75_lengths) if length is not None]

# Plot the 75th percentile average lengths
plt.figure(figsize=(10, 6))
plt.plot(valid_file_indices, valid_lengths, marker='o', linestyle='-', color='b', label='75th Percentile Length')
plt.xlabel('File Number')
plt.ylabel('75th Percentile Average Length (mm)')
plt.title('75th Percentile Average Lengths of Larvae')
plt.ylim(0, 5)
plt.grid()
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Initialize lists to store the 75th percentile and overall average lengths
percentile_75_lengths = []
overall_average_lengths = []

# Loop through all CSV files in the folder
for file_index, file in enumerate(csv_files, start=1):
    # Read the data from the current file
    data = pd.read_csv(file)
    
    # Print the file name for reference
    # print(f"Processing file {file_index}: {file}")
    
    # Calculate the 75th percentile average length
    try:
        average_75th_length_mm = calculate_average_of_75th_percentile(data, exp_settings, step=50)
        percentile_75_lengths.append(average_75th_length_mm)
        print(f"75th Percentile Average Length of Larva in file {file_index}: ~{average_75th_length_mm:.2f} mm")
    except ValueError as e:
        print(f"Error processing file {file_index} for 75th percentile: {e}")
        percentile_75_lengths.append(None)  # Append None for files with errors

    # Calculate the overall average length
    try:
        overall_average_length_mm = calculate_average_length(data, exp_settings, step=50)
        overall_average_lengths.append(overall_average_length_mm)
        print(f"Overall Average Length of Larva in file {file_index}: ~{overall_average_length_mm:.2f} mm")
    except ValueError as e:
        print(f"Error processing file {file_index} for overall average: {e}")
        overall_average_lengths.append(None)  # Append None for files with errors

# Filter out None values for plotting
valid_75th_lengths = [length for length in percentile_75_lengths if length is not None]
valid_overall_lengths = [length for length in overall_average_lengths if length is not None]
valid_file_indices = [i + 1 for i, length in enumerate(percentile_75_lengths) if length is not None]

# Plot the 75th percentile and overall average lengths
plt.figure(figsize=(10, 6))
plt.plot(valid_file_indices, valid_75th_lengths, marker='o', linestyle='-', color='b', label='75th Percentile Length')
plt.plot(valid_file_indices, valid_overall_lengths, marker='x', linestyle='--', color='r', label='Overall Average Length')
plt.xlabel('File Number')
plt.ylabel('Length (mm)')
plt.title('75th Percentile and Overall Average Lengths of Larvae')
plt.ylim(0, 5)  # Set y-axis range from 0 mm to 5 mm
plt.grid()
plt.legend()
plt.show()

In [ ]:
# np.std(lemgthd)

import matplotlib.pyplot as plt
import numpy as np

# Initialize lists to store the 75th percentile lengths, their standard deviations, and overall average lengths
percentile_75_lengths = []
percentile_75_std_devs = []
overall_average_lengths = []

# Loop through all CSV files in the folder
for file_index, file in enumerate(csv_files, start=1):
    # Read the data from the current file
    data = pd.read_csv(file)
    
    # Print the file name for reference
    # print(f"Processing file {file_index}: {file}")
    
    # Calculate the 75th percentile average length and standard deviation
    try:
        lengths_mm = []  # Store all lengths in millimeters for this file
        if 'Pixel per mm' not in exp_settings or exp_settings['Pixel per mm'] <= 0:
            raise ValueError("Invalid 'Pixel per mm' value in exp_settings.")
        ppmm = exp_settings['Pixel per mm']
        for frame_index in range(0, len(data), 50):
            if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
                continue
            x_head = data.loc[frame_index, 'X-Head']
            y_head = data.loc[frame_index, 'Y-Head']
            x_tail = data.loc[frame_index, 'X-Tail']
            y_tail = data.loc[frame_index, 'Y-Tail']
            length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)
            lengths_mm.append(length / ppmm)
        if lengths_mm:
            percentile_75_threshold = np.percentile(lengths_mm, 75)
            lengths_in_75th_percentile = [length for length in lengths_mm if length >= percentile_75_threshold]
            if lengths_in_75th_percentile:
                avg_75th = np.mean(lengths_in_75th_percentile)
                std_75th = np.std(lengths_in_75th_percentile)
                percentile_75_lengths.append(avg_75th)
                percentile_75_std_devs.append(std_75th)
                print(f"75th Percentile Average Length of Larva in file {file_index}: ~{avg_75th:.2f} mm")
            else:
                raise ValueError("No lengths found in the 75th percentile.")
        else:
            raise ValueError("No valid frames to calculate the 75th percentile.")
    except ValueError as e:
        print(f"Error processing file {file_index} for 75th percentile: {e}")
        percentile_75_lengths.append(None)
        percentile_75_std_devs.append(None)

    # Calculate the overall average length
    try:
        overall_average_length_mm = calculate_average_length(data, exp_settings, step=50)
        overall_average_lengths.append(overall_average_length_mm)
        print(f"Overall Average Length of Larva in file {file_index}: ~{overall_average_length_mm:.2f} mm")
    except ValueError as e:
        print(f"Error processing file {file_index} for overall average: {e}")
        overall_average_lengths.append(None)

# Filter out None values for plotting
valid_75th_lengths = [length for length in percentile_75_lengths if length is not None]
valid_75th_std_devs = [std for std in percentile_75_std_devs if std is not None]
valid_overall_lengths = [length for length in overall_average_lengths if length is not None]
valid_file_indices = [i + 1 for i, length in enumerate(percentile_75_lengths) if length is not None]

# Plot the 75th percentile and overall average lengths with error bars
plt.figure(figsize=(10, 6))
plt.errorbar(valid_file_indices, valid_75th_lengths, yerr=valid_75th_std_devs, fmt='o', linestyle='-', color='g', label='75th Percentile Length (with Std Dev)', markersize=3)
plt.plot(valid_file_indices, valid_overall_lengths, marker='x', linestyle='--', color='r', label='Overall Average Length')
plt.xlabel('File Number')
plt.ylabel('Length (mm)')
plt.title('75th Percentile and Overall Average Lengths of Larvae (with Error Bars)')
plt.ylim(0, 5)  # Set y-axis range from 0 mm to 5 mm
plt.grid()
plt.legend()
plt.show()


In [ ]:
import numpy as np

def calculate_angle(data):
    """
    Calculate the angle between the head and the tail of the larva, 
    using the centroid as the axis.

    Parameters:
        data (pd.DataFrame): The DataFrame containing the larva data. 
                             It must include 'X-Head', 'Y-Head', 'X-Tail', 'Y-Tail', 
                             and 'X-Centroid', 'Y-Centroid' columns.

    Returns:
        np.ndarray: An array of angles (in degrees) for each frame.
    """
    # Extract coordinates
    x_head = data['X-Head'].values
    y_head = data['Y-Head'].values
    x_tail = data['X-Tail'].values
    y_tail = data['Y-Tail'].values
    x_centroid = data['X-Centroid'].values
    y_centroid = data['Y-Centroid'].values

    # Calculate vectors from the centroid to the head and tail
    vector_head = np.array([x_head - x_centroid, y_head - y_centroid]).T
    vector_tail = np.array([x_tail - x_centroid, y_tail - y_centroid]).T

    # Normalize the vectors
    norm_head = np.linalg.norm(vector_head, axis=1)
    norm_tail = np.linalg.norm(vector_tail, axis=1)
    unit_vector_head = vector_head / norm_head[:, None]
    unit_vector_tail = vector_tail / norm_tail[:, None]

    # Calculate the dot product and angle
    dot_product = np.einsum('ij,ij->i', unit_vector_head, unit_vector_tail)
    angles = np.arccos(np.clip(dot_product, -1.0, 1.0))  # Clip to avoid numerical errors

    # Convert angles to degrees
    angles_degrees = np.degrees(angles)
    angles = 180 - angles_degrees  # Adjust angle to be between 0 and 180 degrees

    return angles

In [ ]:
angles = calculate_angle(data)
print(angles.min())  # Array of angles for each frame

# Remove NaN values from the angles array
angles_masked = angles[~np.isnan(angles)]

print(angles_masked.min())  # Array of angles without NaN values

In [ ]:
plt.hist(angles_masked)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Initialize lists to store filtered average lengths and file indices
filtered_lengths = []

# Loop through all CSV files in the folder
for file_index, file in enumerate(csv_files, start=1):
    # Read the data from the current file
    data = pd.read_csv(file)
    
    # Calculate angles for the current file
    angles = calculate_angle(data)
    
    # Remove NaN values from angles
    angles = angles[~np.isnan(angles)]
    
    # Filter frames with angles >= 165 degrees
    valid_indices = np.where(angles >= 165)[0]
    
    # Calculate the average length for the filtered frames
    lengths_mm = []
    if 'Pixel per mm' not in exp_settings or exp_settings['Pixel per mm'] <= 0:
        raise ValueError("Invalid 'Pixel per mm' value in exp_settings.")
    ppmm = exp_settings['Pixel per mm']
    for frame_index in valid_indices:
        if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
            continue
        x_head = data.loc[frame_index, 'X-Head']
        y_head = data.loc[frame_index, 'Y-Head']
        x_tail = data.loc[frame_index, 'X-Tail']
        y_tail = data.loc[frame_index, 'Y-Tail']
        length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)
        lengths_mm.append(length / ppmm)
    
    # Calculate the average length for the filtered frames
    if lengths_mm:
        avg_length = np.mean(lengths_mm)
        filtered_lengths.append(avg_length)
        print(f"File {file_index}: Average Length for angles >= 165°: ~{avg_length:.2f} mm")
    else:
        filtered_lengths.append(None)
        print(f"File {file_index}: No valid frames with angles >= 165°.")

# Filter out None values for plotting
valid_lengths = [length for length in filtered_lengths if length is not None]
valid_file_indices = [i + 1 for i, length in enumerate(filtered_lengths) if length is not None]

# Plot the filtered average lengths
plt.figure(figsize=(10, 6))
plt.plot(valid_file_indices, valid_lengths, marker='o', linestyle='-', color='b', label='Filtered Average Length (Angle >= 165°)')
plt.xlabel('File Number')
plt.ylabel('Average Length (mm)')
plt.title('Filtered Average Lengths of Larvae (Angle >= 165°)')
plt.ylim(0, 5)  # Set y-axis range from 0 mm to 5 mm
plt.grid()
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Function to calculate angles
def calculate_angle(data):
    """
    Calculate the angle between the head and the tail of the larva, 
    using the centroid as the axis.

    Parameters:
        data (pd.DataFrame): The DataFrame containing the larva data. 
                             It must include 'X-Head', 'Y-Head', 'X-Tail', 'Y-Tail', 
                             and 'X-Centroid', 'Y-Centroid' columns.

    Returns:
        np.ndarray: An array of angles (in degrees) for each frame.
    """
    # Extract coordinates
    x_head = data['X-Head'].values
    y_head = data['Y-Head'].values
    x_tail = data['X-Tail'].values
    y_tail = data['Y-Tail'].values
    x_centroid = data['X-Centroid'].values
    y_centroid = data['Y-Centroid'].values

    # Calculate vectors from the centroid to the head and tail
    vector_head = np.array([x_head - x_centroid, y_head - y_centroid]).T
    vector_tail = np.array([x_tail - x_centroid, y_tail - y_centroid]).T

    # Normalize the vectors
    norm_head = np.linalg.norm(vector_head, axis=1)
    norm_tail = np.linalg.norm(vector_tail, axis=1)
    unit_vector_head = vector_head / norm_head[:, None]
    unit_vector_tail = vector_tail / norm_tail[:, None]

    # Calculate the dot product and angle
    dot_product = np.einsum('ij,ij->i', unit_vector_head, unit_vector_tail)
    angles = np.arccos(np.clip(dot_product, -1.0, 1.0))  # Clip to avoid numerical errors

    # Convert angles to degrees
    angles_degrees = np.degrees(angles)

    return angles_degrees

# Initialize lists to store the 75th percentile lengths, their standard deviations, and overall average lengths
percentile_75_lengths = []
percentile_75_std_devs = []
overall_average_lengths = []

# Loop through all CSV files in the folder
for file_index, file in enumerate(csv_files, start=1):
    # Read the data from the current file
    data = pd.read_csv(file)
    
    # Calculate the 75th percentile average length and standard deviation
    try:
        lengths_mm = []  # Store all lengths in millimeters for this file
        if 'Pixel per mm' not in exp_settings or exp_settings['Pixel per mm'] <= 0:
            raise ValueError("Invalid 'Pixel per mm' value in exp_settings.")
        ppmm = exp_settings['Pixel per mm']
        for frame_index in range(0, len(data), 50):
            if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
                continue
            x_head = data.loc[frame_index, 'X-Head']
            y_head = data.loc[frame_index, 'Y-Head']
            x_tail = data.loc[frame_index, 'X-Tail']
            y_tail = data.loc[frame_index, 'Y-Tail']
            length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)
            lengths_mm.append(length / ppmm)
        if lengths_mm:
            percentile_75_threshold = np.percentile(lengths_mm, 75)
            lengths_in_75th_percentile = [length for length in lengths_mm if length >= percentile_75_threshold]
            if lengths_in_75th_percentile:
                avg_75th = np.mean(lengths_in_75th_percentile)
                std_75th = np.std(lengths_in_75th_percentile)
                percentile_75_lengths.append(avg_75th)
                percentile_75_std_devs.append(std_75th)
            else:
                raise ValueError("No lengths found in the 75th percentile.")
        else:
            raise ValueError("No valid frames to calculate the 75th percentile.")
    except ValueError as e:
        percentile_75_lengths.append(None)
        percentile_75_std_devs.append(None)

    # Calculate the overall average length
    try:
        overall_average_length_mm = calculate_average_length(data, exp_settings, step=50)
        overall_average_lengths.append(overall_average_length_mm)
    except ValueError as e:
        overall_average_lengths.append(None)

# Filter out None values for plotting
valid_75th_lengths = [length for length in percentile_75_lengths if length is not None]
valid_75th_std_devs = [std for std in percentile_75_std_devs if std is not None]
valid_overall_lengths = [length for length in overall_average_lengths if length is not None]
valid_file_indices = [i + 1 for i, length in enumerate(percentile_75_lengths) if length is not None]

# Initialize lists to store filtered average lengths and file indices
filtered_lengths = []

# Loop through all CSV files in the folder
for file_index, file in enumerate(csv_files, start=1):
    # Read the data from the current file
    data = pd.read_csv(file)
    
    # Calculate angles for the current file
    angles = calculate_angle(data)
    
    # Remove NaN values from angles
    angles = angles[~np.isnan(angles)]
    
    # Filter frames with angles >= 170 degrees
    valid_indices = np.where(angles >= 170)[0]
    
    # Calculate the average length for the filtered frames
    lengths_mm = []
    if 'Pixel per mm' not in exp_settings or exp_settings['Pixel per mm'] <= 0:
        raise ValueError("Invalid 'Pixel per mm' value in exp_settings.")
    ppmm = exp_settings['Pixel per mm']
    for frame_index in valid_indices:
        if data.loc[frame_index, ['X-Head', 'Y-Head', 'X-Tail', 'Y-Tail']].isnull().any():
            continue
        x_head = data.loc[frame_index, 'X-Head']
        y_head = data.loc[frame_index, 'Y-Head']
        x_tail = data.loc[frame_index, 'X-Tail']
        y_tail = data.loc[frame_index, 'Y-Tail']
        length = np.sqrt((x_head - x_tail)**2 + (y_head - y_tail)**2)
        lengths_mm.append(length / ppmm)
    
    # Calculate the average length for the filtered frames
    if lengths_mm:
        avg_length = np.mean(lengths_mm)
        filtered_lengths.append(avg_length)
    else:
        filtered_lengths.append(None)

# Filter out None values for plotting
valid_lengths = [length for length in filtered_lengths if length is not None]
valid_file_indices_filtered = [i + 1 for i, length in enumerate(filtered_lengths) if length is not None]

# Plot the filtered average lengths and 75th percentile lengths on the same graph
plt.figure(figsize=(10, 6))

# Plot filtered average lengths (Angle >= 170°)
plt.plot(valid_file_indices_filtered, valid_lengths, marker='o', linestyle='-', color='b', label='Filtered Average Length (Angle >= 170°)')

# Plot 75th percentile lengths
plt.errorbar(valid_file_indices, valid_75th_lengths, yerr=valid_75th_std_devs, fmt='o', linestyle='-', color='g', label='75th Percentile Length (with Std Dev)', markersize=3)

# Add labels, title, and legend
plt.xlabel('File Number')
plt.ylabel('Length (mm)')
plt.title('Filtered Average Lengths (Angle >= 170°) and 75th Percentile Lengths')
plt.ylim(0, 5)  # Set y-axis range from 0 mm to 5 mm
plt.grid()
plt.legend()

# Show the plot
plt.show()

In [ ]:
import cv2
img = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
data = pd.read_csv(img)

In [ ]:
print(img)

In [ ]:
import numpy as np
from scipy.ndimage import rotate
import matplotlib.pyplot as plt

# Rotate the matrix by an arbitrary angle (e.g., 45 degrees)
rotated_matrix = rotate(data, angle=45, reshape=True)

# Display the original and rotated matrices
print("Original Matrix:")
print(data)

print("\nRotated Matrix (45 degrees):")
print(rotated_matrix)

# Visualize the rotation
plt.subplot(1, 2, 1)
plt.title("Original")
plt.imshow(data, cmap='gray')

plt.subplot(1, 2, 2)
plt.title("Rotated (45°)")
plt.imshow(rotated_matrix, cmap='gray')

plt.show()

In [ ]:
plt.imshow(rotated_matrix)
plt.show()
plt.imshow(data)
plt.xlim(100, 600)
plt.ylim(400, 0)
plt.show()


Matching 2 Arenas.

In [ ]:
img1 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
img2 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-37-15_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'

img1_pandas = pd.read_csv(img1)
img2_pandas = pd.read_csv(img2)
plt.title("1")
plt.imshow(img1_pandas, cmap='gray')
plt.show()
plt.title("2")
plt.imshow(img2_pandas, cmap='gray')
plt.show()
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Image 1")
plt.imshow(img1_pandas, cmap='gray')

plt.subplot(1, 2, 2)
plt.title("Image 2")
plt.imshow(img2_pandas, cmap='gray')

plt.tight_layout()
plt.show()
file2 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-37-15_PiVR4_snapshot_recenter/2025.05.05_18-37-15_data_level1.csv'
file1 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/2025.05.05_18-31-52_data_level1.csv'
data1 = pd.read_csv(file1)
data2 = pd.read_csv(file2)

# these are the first two files, print the starting positions of the larvae
position1 = data1.loc[0, ['X-Centroid', 'Y-Centroid']]
position2 = data2.loc[0, ['X-Centroid', 'Y-Centroid']]
print(f"Position in file 1:\n{position1}\n")
print(f"Position in file 2:\n {position2}\n")

In [ ]:
#translating the second one to have the same center as the first one
# Calculate the translation offsets
x_offset = position1['X-Centroid'] - position2['X-Centroid']
y_offset = position1['Y-Centroid'] - position2['Y-Centroid']

# Apply the translation to the second image
translated_img2 = img2_pandas.shift(int(y_offset), axis=0).shift(int(x_offset), axis=1)

# Fill NaN values introduced by the shift with 0
translated_img2 = translated_img2.fillna(0)

# Visualize the translated image
plt.imshow(translated_img2, cmap='gray')
plt.title("Translated Image 2")
# Show the original Image 2 for comparison
plt.figure(figsize=(10, 5))

# Original Image 2
plt.subplot(1, 2, 1)
plt.imshow(img2_pandas, cmap='gray')
plt.title("Original Image 2")

# Translated Image 2
plt.subplot(1, 2, 2)
plt.imshow(translated_img2, cmap='gray')
plt.title("Translated Image 2")

plt.tight_layout()
plt.show()

# Plot both images on the same graph for comparison
plt.figure(figsize=(10, 5))

# Image 1
plt.subplot(1, 2, 1)
plt.imshow(img1_pandas, cmap='gray')
plt.title("Image 1")

# Translated Image 2
plt.subplot(1, 2, 2)
plt.imshow(translated_img2, cmap='gray')
plt.title("Translated Image 2")

plt.tight_layout()
plt.show()

# Necessary functions

## rotate_image

In [ ]:
import numpy as np
from scipy.ndimage import rotate

import matplotlib.pyplot as plt

def rotate_image(image, angle, center):
    """
    Rotate an image counterclockwise about a specified center point.

    Parameters:
        image (pd.DataFrame): The image to rotate (as a pandas DataFrame).
        angle (float): The angle in degrees to rotate the image counterclockwise.
        center (tuple): The (x, y) coordinates of the center of rotation.

    Returns:
        np.ndarray: The rotated image.
    """
    # Convert the DataFrame to a NumPy array
    image_array = image.to_numpy()

    # Translate the image so the center of rotation is at the origin
    shift_x, shift_y = center[0] - image_array.shape[1] // 2, center[1] - image_array.shape[0] // 2
    shifted_image = np.roll(image_array, shift=(-int(shift_y), -int(shift_x)), axis=(0, 1))

    # Rotate the image
    rotated_image = rotate(shifted_image, angle=angle, reshape=False)

    # Translate the image back to its original position
    rotated_image = np.roll(rotated_image, shift=(int(shift_y), int(shift_x)), axis=(0, 1))

    return rotated_image

# # Example usage:
# # Adjustable angle
# angle = 83  # Change this value to rotate by a different angle

# # Center of rotation
# center = (598.192, 439.824)

# # Rotate the image
# rotated_translated_img2 = rotate_image(translated_img2, angle, center)

# # Visualize the rotated image
# plt.imshow(rotated_translated_img2, cmap='gray')
# plt.title(f"Rotated Image (Angle: {angle}°)")
# # Show the rotated image and the original Image 1 side by side for comparison
# plt.figure(figsize=(12, 6))

# # Original Image 1
# plt.subplot(1, 2, 1)
# plt.imshow(img1_pandas, cmap='gray')
# plt.title("Image 1")

# # Rotated Image 2
# plt.subplot(1, 2, 2)
# plt.imshow(rotated_translated_img2, cmap='gray')
# plt.title(f"Rotated and Translated Image 2 (Angle: {angle}°)")

# plt.tight_layout()
# plt.show()

In [ ]:
# import numpy as np
# import pandas as pd
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# # Load the images as NumPy arrays
# img1 = pd.read_csv('/path/to/image1.csv').to_numpy()
# img2 = pd.read_csv('/path/to/image2.csv').to_numpy()

# # Convert to uint8 (required for OpenCV functions)
# img1 = (img1 / img1.max() * 255).astype(np.uint8)
# img2 = (img2 / img2.max() * 255).astype(np.uint8)

# # Use img1 as the template and search for it in img2
# result = cv2.matchTemplate(img2, img1, method=cv2.TM_CCOEFF_NORMED)

# # Find the best match location
# min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)

# # Extract the top-left corner of the matched region
# top_left = max_loc
# print(f"Best match top-left corner: {top_left}")

# # Translate img2 to align with img1
# translation = (top_left[1], top_left[0])  # (y, x)
# aligned_img2 = np.roll(img2, shift=translation, axis=(0, 1))

# # Visualize the alignment
# plt.figure(figsize=(10, 5))

# plt.subplot(1, 3, 1)
# plt.title("Image 1 (Template)")
# plt.imshow(img1, cmap='gray')

# plt.subplot(1, 3, 2)
# plt.title("Aligned Image 2")
# plt.imshow(aligned_img2, cmap='gray')

# plt.subplot(1, 3, 3)
# plt.title("Difference")
# plt.imshow(img1 - aligned_img2, cmap='gray')

# plt.tight_layout()
# plt.show()
# import matplotlib.pyplot as plt

# # Load the images as pandas DataFrames
# img1_pandas = pd.read_csv(img1).to_numpy()
# img2_pandas = pd.read_csv(img2).to_numpy()

# # Step 1: Initial translation using centroid positions
# position1 = data1.loc[0, ['X-Centroid', 'Y-Centroid']] # Replace with actual centroid of img1
# position2 = data2.loc[0, ['X-Centroid', 'Y-Centroid']] 
# difference = np.array(position1) - np.array(position2)
# translation = position2 - difference

# # Apply initial translation
# translated_img2 = shift(img2_pandas, shift=(translation[1], translation[0]), mode='constant', cval=0)

# # Step 2: Refine alignment using cross-correlation
# shift_yx, error, diffphase = register_translation(img1_pandas, translated_img2, upsample_factor=10)
# refined_img2 = shift(translated_img2, shift=(shift_yx[0], shift_yx[1]), mode='constant', cval=0)

# # Step 3: Rotation adjustment (if needed)
# # Calculate rotation angle (e.g., using feature matching or manual estimation)
# rotation_angle = 0  # Replace with the calculated angle
# rotated_img2 = rotate(refined_img2, angle=rotation_angle, reshape=False)

# # Step 4: Visualize the alignment
# plt.figure(figsize=(10, 5))

# plt.subplot(1, 3, 1)
# plt.title("Image 1")
# plt.imshow(img1_pandas, cmap='gray')

# plt.subplot(1, 3, 2)
# plt.title("Aligned Image 2")
# plt.imshow(rotated_img2, cmap='gray')

# plt.subplot(1, 3, 3)
# plt.title("Difference")
# plt.imshow(img1_pandas - rotated_img2, cmap='gray')

# plt.tight_layout()
# plt.show()

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

# Convert img1_pandas to a NumPy array
img1_array = img1_pandas.to_numpy()

# Ensure both images have the same width by padding the smaller one
width1 = img1_array.shape[1]
width2 = rotated_translated_img2.shape[1]

if width1 > width2:
    padding = ((0, 0), (0, width1 - width2))
    rotated_translated_img2_padded = np.pad(rotated_translated_img2, padding, mode='constant', constant_values=0)
    stacked_images = np.vstack((img1_array, rotated_translated_img2_padded))
elif width2 > width1:
    padding = ((0, 0), (0, width2 - width1))
    img1_padded = np.pad(img1_array, padding, mode='constant', constant_values=0)
    stacked_images = np.vstack((img1_padded, rotated_translated_img2))
else:
    stacked_images = np.vstack((img1_array, rotated_translated_img2))

# Visualize the stacked images
plt.imshow(stacked_images, cmap='gray')
plt.title("Stacked Images (Image 1 on top of Rotated and Translated Image 2)")
plt.show()

## align_images1

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def align_images1(img1_path, img2_path):
    """
    Aligns two images using ORB feature matching and homography.

    Parameters:
        img1_path (str): File path to the first image (reference image).
        img2_path (str): File path to the second image (image to be aligned).

    Returns:
        np.ndarray: The aligned version of the second image.
    """
    # Load the images as NumPy arrays
    img1 = pd.read_csv(img1_path).to_numpy()
    img2 = pd.read_csv(img2_path).to_numpy()

    # Convert to uint8 (required for OpenCV functions)
    img1 = (img1 / img1.max() * 255).astype(np.uint8)
    img2 = (img2 / img2.max() * 255).astype(np.uint8)

    # Initialize ORB detector
    orb = cv2.ORB_create()

    # Detect keypoints and descriptors
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    # Match features using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Compute homography matrix
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

    # Warp img2 to align with img1
    aligned_img2 = cv2.warpPerspective(img2, H, (img1.shape[1], img1.shape[0]))

    return aligned_img2



# img1p = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
# img2p = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-37-15_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'

# # Example usage:
# # Show the first image
# img1 = pd.read_csv(img1p).to_numpy()
# plt.imshow(img1, cmap='gray')
# plt.title("First Image")
# plt.show()

# # Align the second image to the first
# aligned_image = align_images(img1p, img2p)
# plt.imshow(aligned_image, cmap='gray')
# plt.title("Aligned Image")
# plt.show()


## align_images2 (works for snapshot experiments)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def align_images2(img1_path, img2_path):
    """
    Aligns two images using ORB feature matching and homography.

    Parameters:
        img1_path (str): File path to the first image (reference image).
        img2_path (str): File path to the second image (image to be aligned).

    Returns:
        tuple: (aligned_img2, rotation_angle, translation)
            - aligned_img2 (np.ndarray): The aligned version of the second image.
            - rotation_angle (float): The rotation angle in degrees.
            - translation (tuple): The translation (x, y) in pixels.
    """
    # Load the images as NumPy arrays
    img1 = pd.read_csv(img1_path).to_numpy()
    img2 = pd.read_csv(img2_path).to_numpy()

    # Convert to uint8 (required for OpenCV functions)
    img1 = (img1 / img1.max() * 255).astype(np.uint8)
    img2 = (img2 / img2.max() * 255).astype(np.uint8)

    # Preprocess images
    img1 = cv2.GaussianBlur(img1, (5, 5), 0)
    img2 = cv2.GaussianBlur(img2, (5, 5), 0)

    # Initialize ORB detector
    orb = cv2.ORB_create(nfeatures=100)

    # Detect keypoints and descriptors
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    # Check if descriptors are valid
    if des1 is None or des2 is None:
        print(f"Skipping alignment: Failed to detect keypoints or descriptors in {img1_path} or {img2_path}.")
        return None, None, None

    # Match features using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Compute homography matrix
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

    # Extract translation
    tx = H[0, 2]  # x-translation
    ty = H[1, 2]  # y-translation

    # Extract rotation angle
    scale = np.sqrt(H[0, 0]**2 + H[1, 0]**2)
    R = H[:2, :2] / scale
    rotation_angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))

    # Warp img2 to align with img1
    aligned_img2 = cv2.warpPerspective(img2, H, (img1.shape[1], img1.shape[0]))

    return aligned_img2, rotation_angle, (tx, ty)

## Example usage
# img1p = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
# img2p = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-37-15_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'

# # Show the first image
# img1 = pd.read_csv(img1p).to_numpy()
# plt.imshow(img1, cmap='gray')
# plt.title("First Image")
# plt.show()

# # Align the second image to the first
# aligned_image, rot_ang, trans = align_images(img1p, img2p)
# plt.imshow(aligned_image, cmap='gray')
# plt.title("Aligned Image")
# plt.show()

# print(f"Rotation Angle: {rot_ang} degrees")
# print(f"Translation: {trans} pixels")

## align images sift (for average plume experiments)

In [ ]:
import cv2
import numpy as np
import pandas as pd

def align_images_sift(img1_path, img2_path):
    """
    Aligns two images using SIFT feature matching and homography.

    Parameters:
        img1_path (str): File path to the first image (reference image).
        img2_path (str): File path to the second image (image to be aligned).

    Returns:
        tuple: (aligned_img2, rotation_angle, translation)
            - aligned_img2 (np.ndarray): The aligned version of the second image.
            - rotation_angle (float): The rotation angle in degrees.
            - translation (tuple): The translation (x, y) in pixels.
    """
    # Load the images as NumPy arrays
    img1 = pd.read_csv(img1_path).to_numpy()
    img2 = pd.read_csv(img2_path).to_numpy()

    # Convert to uint8 (required for OpenCV functions)
    img1 = (img1 / img1.max() * 255).astype(np.uint8)
    img2 = (img2 / img2.max() * 255).astype(np.uint8)

    # Initialize SIFT detector
    sift = cv2.SIFT_create(nfeatures = 100)

    # Detect keypoints and descriptors
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    # Check if descriptors are valid
    if des1 is None or des2 is None:
        print(f"Skipping alignment: Failed to detect keypoints or descriptors in {img1_path} or {img2_path}.")
        return None, None, None

    # Match features using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Compute homography matrix
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 10.0)
    # print(f"Homography matrix:\n{H}")

    # Extract translation
    tx = H[0, 2]  # x-translation
    ty = H[1, 2]  # y-translation
    print(f"tx: {tx}, ty: {ty}")

    # Extract rotation angle
    scale = np.sqrt(H[0, 0]**2 + H[1, 0]**2)
    R = H[:2, :2] / scale
    rotation_angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))

    # Warp img2 to align with img1
    aligned_img2 = cv2.warpPerspective(img2, H, (img1.shape[1], img1.shape[0]))

    return aligned_img2, rotation_angle, (tx, ty)

In [ ]:
import cv2
import numpy as np
import pandas as pd

def align_images_sift2(img1_path, img2_path):
    """
    Aligns two images using SIFT feature matching and homography with improved accuracy.

    Parameters:
        img1_path (str): File path to the first image (reference image).
        img2_path (str): File path to the second image (image to be aligned).

    Returns:
        tuple: (aligned_img2, rotation_angle, translation)
            - aligned_img2 (np.ndarray): The aligned version of the second image.
            - rotation_angle (float): The rotation angle in degrees.
            - translation (tuple): The translation (x, y) in pixels.
    """
    # Load the images as NumPy arrays
    img1 = pd.read_csv(img1_path).to_numpy()
    img2 = pd.read_csv(img2_path).to_numpy()

    # Convert to uint8 (required for OpenCV functions)
    img1 = (img1 / img1.max() * 255).astype(np.uint8)
    img2 = (img2 / img2.max() * 255).astype(np.uint8)

    # Preprocess images
    img1 = cv2.GaussianBlur(img1, (5, 5), 0)
    img2 = cv2.GaussianBlur(img2, (5, 5), 0)

    # Initialize SIFT detector
    sift = cv2.SIFT_create()

    # Detect keypoints and descriptors
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    # Check if descriptors are valid
    if des1 is None or des2 is None:
        print(f"Skipping alignment: Failed to detect keypoints or descriptors in {img1_path} or {img2_path}.")
        return None, None, None

    # Match features using BFMatcher with Lowe's ratio test
    bf = cv2.BFMatcher(cv2.NORM_L2)
    matches = bf.knnMatch(des1, des2, k=2)
    good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Compute homography matrix using RHO
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RHO)
    if H is None or np.linalg.matrix_rank(H) < 3:
        print("Invalid homography matrix.")
        return None, None, None

    # Extract translation
    tx = H[0, 2]  # x-translation
    ty = H[1, 2]  # y-translation

    # Extract rotation angle
    scale = np.sqrt(H[0, 0]**2 + H[1, 0]**2)
    R = H[:2, :2] / scale
    rotation_angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))

    # Warp img2 to align with img1
    aligned_img2 = cv2.warpPerspective(img2, H, (img1.shape[1], img1.shape[0]))

    return aligned_img2, rotation_angle, (tx, ty)

In [ ]:
import cv2
import numpy as np
import pandas as pd

def align_images_sift_lmeds(img1_path, img2_path):
    """
    Aligns two images using SIFT feature matching and homography with LMEDS.

    Parameters:
        img1_path (str): File path to the first image (reference image).
        img2_path (str): File path to the second image (image to be aligned).

    Returns:
        tuple: (aligned_img2, rotation_angle, translation)
            - aligned_img2 (np.ndarray): The aligned version of the second image.
            - rotation_angle (float): The rotation angle in degrees.
            - translation (tuple): The translation (x, y) in pixels.
    """
    # Load the images as NumPy arrays
    img1 = pd.read_csv(img1_path).to_numpy()
    img2 = pd.read_csv(img2_path).to_numpy()

    # Convert to uint8 (required for OpenCV functions)
    img1 = (img1 / img1.max() * 255).astype(np.uint8)
    img2 = (img2 / img2.max() * 255).astype(np.uint8)

    # Initialize SIFT detector
    sift = cv2.SIFT_create()

    # Detect keypoints and descriptors
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    # Check if descriptors are valid
    if des1 is None or des2 is None:
        print(f"Skipping alignment: Failed to detect keypoints or descriptors in {img1_path} or {img2_path}.")
        return None, None, None

    # Match features using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Compute homography matrix using LMEDS
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.LMEDS)
    if H is None:
        print("Failed to compute homography matrix.")
        return None, None, None

    # Extract translation
    tx = H[0, 2]  # x-translation
    ty = H[1, 2]  # y-translation

    # Extract rotation angle
    scale = np.sqrt(H[0, 0]**2 + H[1, 0]**2)
    R = H[:2, :2] / scale
    rotation_angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))

    # Warp img2 to align with img1
    aligned_img2 = cv2.warpPerspective(img2, H, (img1.shape[1], img1.shape[0]))

    return aligned_img2, rotation_angle, (tx, ty)

In [ ]:
import cv2
import numpy as np
import pandas as pd

def align_images_rho(img1_path, img2_path):
    """
    Aligns two images using ORB feature matching and homography with RHO.

    Parameters:
        img1_path (str): File path to the first image (reference image).
        img2_path (str): File path to the second image (image to be aligned).

    Returns:
        tuple: (aligned_img2, rotation_angle, translation)
            - aligned_img2 (np.ndarray): The aligned version of the second image.
            - rotation_angle (float): The rotation angle in degrees.
            - translation (tuple): The translation (x, y) in pixels.
    """
    # Load the images as NumPy arrays
    img1 = pd.read_csv(img1_path).to_numpy()
    img2 = pd.read_csv(img2_path).to_numpy()

    # Convert to uint8 (required for OpenCV functions)
    img1 = (img1 / img1.max() * 255).astype(np.uint8)
    img2 = (img2 / img2.max() * 255).astype(np.uint8)

    # Initialize ORB detector
    orb = cv2.ORB_create()

    # Detect keypoints and descriptors
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    # Check if descriptors are valid
    if des1 is None or des2 is None:
        print(f"Skipping alignment: Failed to detect keypoints or descriptors in {img1_path} or {img2_path}.")
        return None, None, None

    # Match features using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Compute homography matrix using RHO
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RHO)
    if H is None:
        print("Failed to compute homography matrix.")
        return None, None, None

    # Extract translation
    tx = H[0, 2]  # x-translation
    ty = H[1, 2]  # y-translation

    # Extract rotation angle
    scale = np.sqrt(H[0, 0]**2 + H[1, 0]**2)
    R = H[:2, :2] / scale
    rotation_angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))

    # Warp img2 to align with img1
    aligned_img2 = cv2.warpPerspective(img2, H, (img1.shape[1], img1.shape[0]))

    return aligned_img2, rotation_angle, (tx, ty)


## align_all_images_in_folders

In [ ]:
import os
import pandas as pd
import json
import matplotlib.pyplot as plt

def align_all_images_in_folders(base_folder):
    """
    Loops through all folders in a base directory, aligns all image files in each folder
    to the first image file using the align_images() function, and saves the aligned images.
    Updates the experiment_settings.json file in each folder with rotation and translation values.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with image files.

    Returns:
        None
    """
    all_images = []
    all_folders = []

    # Check if the base folder exists
    if not os.path.exists(base_folder):
        print(f"Base folder {base_folder} does not exist.")
        return
    
    # Loop through all subfolders in the base folder
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if os.path.isdir(folder_path):
            # Get all CSV image files in the folder
            image_files = [f for f in os.listdir(folder_path) if f.endswith('].csv') and not f.startswith('aligned_')]
            if len(image_files) < 1:
                print(f"Skipping folder {folder} (not enough image files).")
                continue

            all_folders.append(folder_path)
            all_images.append(image_files[0])

    # Ensure there are valid images to process
    if len(all_images) == 0:
        print("No valid image files found in the base folder.")
        return

    # Use the first image file as the reference
    reference_image_path = os.path.join(all_folders[0], all_images[0])

    # Align all other images to the reference image
    for image_index, image_file in enumerate(all_images):
        image_path = os.path.join(all_folders[image_index], image_file)
        
        # Call the align_images function
        if "average" in base_folder.lower():
            aligned_image, rotation_angle, translation = align_images_sift(reference_image_path, image_path)

        elif "snapshot" in base_folder.lower():
            aligned_image, rotation_angle, translation = align_images2(reference_image_path, image_path)
        
        # Skip if alignment fails
        if aligned_image is None or rotation_angle is None or translation is None:
            print(f"Skipping {image_file} due to alignment failure.")
            continue

        # Save the aligned image
        aligned_image_path = os.path.join(all_folders[image_index], f"aligned_{os.path.basename(image_file)}")
        pd.DataFrame(aligned_image).to_csv(aligned_image_path, index=False, header=False)
        print(f"Aligned and saved: {aligned_image_path}")

        # Save a new JSON file with updated settings
        json_path = os.path.join(all_folders[image_index], "experiment_settings.json")
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                settings = json.load(f)
            
            # Add rotation and translation values
            settings['Rotation Angle (degrees)'] = rotation_angle
            settings['Translation (pixels)'] = {'x': translation[0], 'y': translation[1]}

            # Save the updated JSON file with a new name
            new_json_path = os.path.join(all_folders[image_index], f"updated_{os.path.basename(json_path)}")
            with open(new_json_path, 'w') as f:
                json.dump(settings, f, indent=4)
            print(f"Saved updated JSON file: {new_json_path}")
        else:
            print(f"JSON file not found in folder: {all_folders[image_index]}")

# Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/average_plume'
# align_all_images_in_folders(base_folder)

## show aligned images in folder (check)

In [ ]:
def show_aligned_images(base_folder_path):
    """
    Loops through all folders in the base directory, finds aligned image files,
    and displays them using matplotlib.
    Parameters:
        base_folder_path (str): Path to the base folder containing subfolders with aligned image files.
    Returns:
        None
    """
    for folder in os.listdir(base_folder_path):
        folder_path = os.path.join(base_folder_path, folder)
        if os.path.isdir(folder_path):
            # Get all CSV image files in the folder
            aligned_image_files = [f for f in os.listdir(folder_path) if f.startswith('aligned_P') and f.endswith('].csv')]
            print(aligned_image_files)

            for aligned_file in aligned_image_files:
                aligned_file_path = os.path.join(folder_path, aligned_file)
                print(f"Aligned image file: {aligned_file}")
                
                # Load and display the aligned image
                aligned_image = pd.read_csv(aligned_file_path).to_numpy()
                plt.imshow(aligned_image, cmap='gray')
                plt.title(f"Aligned Image: {aligned_file}, Located in {folder}")
                plt.show()


## rotate_and_translate

In [ ]:
import os
import pandas as pd
import numpy as np
import json

def rotate_and_translate(x, y, angle, tx, ty):
    """
    Rotate and translate coordinates.

    Parameters:
        x (float): X-coordinate.
        y (float): Y-coordinate.
        angle (float): Rotation angle in degrees.
        tx (float): Translation in the X direction.
        ty (float): Translation in the Y direction.

    Returns:
        tuple: Transformed (x, y) coordinates.
    """
    # Handle NaN values
    if pd.isna(x) or pd.isna(y):
        print(f"Invalid input: x={x}, y={y}")
        return np.nan, np.nan

    # Convert angle to radians
    angle_rad = np.radians(angle)
    

    # Apply rotation
    x_rot = x * np.cos(angle_rad) - y * np.sin(angle_rad)
    y_rot = x * np.sin(angle_rad) + y * np.cos(angle_rad)
    

    # Apply translation
    x_trans = x_rot + tx
    y_trans = y_rot + ty
    

    return x_trans, y_trans

# Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/average_plume'
# process_gui_corrected_files(base_folder)

## process_gui_corrected_files

In [ ]:
def process_gui_corrected_files(base_folder):
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue

        gui_corrected_files = [
            f for f in os.listdir(folder_path)
            if f.endswith("GUIcorrected.csv") and not f.startswith("aligned_")
        ]
        if not gui_corrected_files:
            print(f"No valid GUIcorrected.csv file found in {folder_path}. Skipping.")
            continue

        gui_corrected_file = gui_corrected_files[0]
        gui_corrected_path = os.path.join(folder_path, gui_corrected_file)

        json_path = os.path.join(folder_path, "updated_experiment_settings.json")
        if not os.path.exists(json_path):
            print(f"No experiment_settings.json file found in {folder_path}. Skipping.")
            continue

        with open(json_path, "r") as f:
            settings = json.load(f)

        rotation_angle = settings.get("Rotation Angle (degrees)", 0)
        translation = settings.get("Translation (pixels)", {"x": 0, "y": 0})
        tx = translation.get("x", 0)
        ty = translation.get("y", 0)

        print(f"Processing {gui_corrected_file} with Rotation: {rotation_angle}, Translation: (tx={tx}, ty={ty})")

        data = pd.read_csv(gui_corrected_path)
        data = data.apply(pd.to_numeric, errors='coerce').fillna(0)

        for col_x, col_y in [("X-Centroid", "Y-Centroid"),
                             ("X-Head", "Y-Head"),
                             ("X-Tail", "Y-Tail"),
                             ("X-Midpoint", "Y-Midpoint")]:
            if col_x in data.columns and col_y in data.columns:
                data[[col_x, col_y]] = data.apply(
                    lambda row: rotate_and_translate(row[col_x], row[col_y], rotation_angle, tx, ty),
                    axis=1, result_type="expand"
                )
            else:
                print(f"Columns {col_x} and/or {col_y} not found in {gui_corrected_file}. Skipping transformation for these columns.")

        aligned_file_name = f"aligned_{gui_corrected_file}"
        aligned_file_path = os.path.join(folder_path, aligned_file_name)
        data.to_csv(aligned_file_path, index=False)
        print(f"Saved aligned data to {aligned_file_path}")


## Not necessary

### plot_larva_path (not necessary) 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_larva_path(file_path):
    """
    Plot the path of the larva using the midpoint at each frame.

    Parameters:
        file_path (str): Path to the CSV file (aligned or original).

    Returns:
        None
    """
    # Load the data
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    data = pd.read_csv(file_path)

    # Check if required columns exist
    if 'X-Midpoint' not in data.columns or 'Y-Midpoint' not in data.columns:
        print("Required columns 'X-Midpoint' and 'Y-Midpoint' not found in the file.")
        return
    
    # Exclude the first and last 30 frames
    data = data.iloc[30:-30]

    # Extract the midpoint coordinates
    x_midpoint = data['X-Midpoint']
    y_midpoint = data['Y-Midpoint']

    # Plot the path
    plt.figure(figsize=(8, 8))
    plt.plot(x_midpoint, y_midpoint, marker='o', markersize=2, linestyle='-', label='Larva Path')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.title('Larva Path (Midpoint Trajectory)')
    plt.legend()
    plt.axis('equal')
    plt.grid()
    plt.show()

# # Example usage
# file_path = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/aligned_2025.05.05_18-31-52_data_GUIcorrected.csv'  # Replace with the actual file path
# plot_larva_path(file_path)
# # Load and display the aligned image
# aligned_image_path1 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/aligned_PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
# aligned_image1 = pd.read_csv(aligned_image_path1).to_numpy()
# plt.imshow(aligned_image1, cmap='gray')
# plt.ylim(0,aligned_image1.shape[0])
# plt.title(f"Aligned Image:")
# plt.show()

# file_path2  = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/2025.05.05_18-31-52_data_GUIcorrected.csv'
# plot_larva_path(file_path2)
# aligned_image_path2 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
# aligned_image2 = pd.read_csv(aligned_image_path2).to_numpy()
# plt.imshow(aligned_image2, cmap='gray')
# plt.ylim(0,aligned_image2.shape[0])
# plt.title(f"Original Image: ")
# plt.show()



## plot_all_aligned_paths1

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_all_aligned_paths1(base_folder):
    """
    Plot all aligned larva paths from a folder of folders on a single graph.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with aligned CSV files.

    Returns:
        None
    """
    # Initialize the plot
    plt.figure(figsize=(10, 10))
    
    # Loop through all subfolders in the base folder
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue
        
        # Find all aligned CSV files in the subfolder
        aligned_files = [f for f in os.listdir(folder_path) if f.startswith('aligned_2025') and f.endswith('.csv')]
        for aligned_file in aligned_files:
            aligned_file_path = os.path.join(folder_path, aligned_file)
            
            # Load the data
            data = pd.read_csv(aligned_file_path)
            
            # Check if required columns exist
            if 'X-Midpoint' not in data.columns or 'Y-Midpoint' not in data.columns:
                print(f"Required columns not found in {aligned_file}. Skipping.")
                continue
            
            # Exclude the first and last 30 frames only if there are enough rows
            if len(data) > 60:
                data = data.iloc[30:-30]
            else:
                print(f"File {aligned_file} has fewer than 60 rows. Skipping frame exclusion.")
            
            # Remove rows with invalid X-Midpoint or Y-Midpoint values
            data = data.dropna(subset=['X-Midpoint', 'Y-Midpoint'])
            data = data[(data['X-Midpoint'] != 0) & (data['Y-Midpoint'] != 0)]
            
            # Extract the midpoint coordinates
            x_midpoint = data['X-Midpoint']
            y_midpoint = data['Y-Midpoint']
            
            # Plot the path
            plt.plot(x_midpoint, y_midpoint, label=aligned_file, alpha=0.6)
    
    # Finalize the plot
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.title('All Aligned Larva Paths')
    plt.legend(loc='upper right', fontsize='small', bbox_to_anchor=(1.3, 1))
    plt.axis('equal')
    plt.grid()
    plt.show()

# # Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot'  # Replace with your base folder path
# plot_all_aligned_paths(base_folder)

## plot_all_aligned_paths2

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_all_aligned_paths2(base_folder):
    """
    Plot all aligned larva paths from a folder of folders on a single graph.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with aligned CSV files.

    Returns:
        None
    """
    # Initialize the plot
    plt.figure(figsize=(10, 10))
    
    # Loop through all subfolders in the base folder
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue
        
        # Find all aligned CSV files in the subfolder
        aligned_files = [f for f in os.listdir(folder_path) if f.startswith('aligned_2025') and f.endswith('.csv')]
        for aligned_file in aligned_files:
            aligned_file_path = os.path.join(folder_path, aligned_file)
            
            # Load the data
            data = pd.read_csv(aligned_file_path)
            
            # Check if required columns exist
            required_columns = ['X-Midpoint', 'Y-Midpoint', 'Ymin-bbox', 'Ymax-bbox']
            if not all(col in data.columns for col in required_columns):
                print(f"Required columns not found in {aligned_file}. Skipping.")
                continue
            
            # Exclude the first and last 30 frames only if there are enough rows
            if len(data) > 60:
                data = data.iloc[30:-30]
            else:
                print(f"File {aligned_file} has fewer than 60 rows. Skipping frame exclusion.")
            
            # Remove rows with invalid X-Midpoint, Y-Midpoint, Ymin-bbox, or Ymax-bbox values
            data = data.dropna(subset=['X-Midpoint', 'Y-Midpoint', 'Ymin-bbox', 'Ymax-bbox'])
            data = data[(data['X-Midpoint'] > 0) & (data['Y-Midpoint'] > 0)]
            data = data[(data['Ymin-bbox'] != 0.0) & (data['Ymax-bbox'] != 0.0)]

            # Check for jumps in position
            x_midpoint = data['X-Midpoint'].values
            y_midpoint = data['Y-Midpoint'].values
            distances = np.sqrt(np.diff(x_midpoint)**2 + np.diff(y_midpoint)**2)
            if np.any(distances > 20):
                jump_index = np.where(distances > 20)[0][0] + 1  # Get the first jump index
                data = data.iloc[:jump_index]  # Exclude frames after the jump
            
            # Extract the midpoint coordinates
            x_midpoint = data['X-Midpoint']
            y_midpoint = data['Y-Midpoint']
            
            # Plot the path
            plt.plot(x_midpoint, y_midpoint, label=aligned_file, alpha=0.6)
    
    # Finalize the plot
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.title('All Aligned Larva Paths')
    plt.legend(loc='upper right', fontsize='small', bbox_to_anchor=(1.3, 1))
    plt.axis('equal')
    plt.grid()
    plt.show()

# # Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot'  # Replace with your base folder path
# plot_all_aligned_paths(base_folder)

## final_plot_all_good_aligned_paths

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def final_plot_all_good_aligned_paths(base_folder, stimulation_threshold=9950):
    """
    Plot all aligned larva paths from a folder of folders on a single graph.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with aligned CSV files.

    Returns:
        None
    """
    # Initialize the plot
    plt.figure(figsize=(10, 10))
    
    # Loop through all subfolders in the base folder
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue
        
        # Find all aligned CSV files in the subfolder
        aligned_files = [f for f in os.listdir(folder_path) if f.startswith('aligned_2025') and f.endswith('.csv')]
        for aligned_file in aligned_files:
            aligned_file_path = os.path.join(folder_path, aligned_file)
            
            # Load the data
            data = pd.read_csv(aligned_file_path)
            
            # Check if required columns exist
            required_columns = ['X-Midpoint', 'Y-Midpoint', 'Ymin-bbox', 'Ymax-bbox']
            if not all(col in data.columns for col in required_columns):
                print(f"Required columns not found in {aligned_file}. Skipping.")
                continue
            
            # # Check if "stimulation" is 0.0 for more than half the frames
            # if (data['stimulation'] == 0.0).sum() > len(data) / 4:
            #     print(f"File {aligned_file} excluded due to stimulation being 0.0 for more than half the frames.")
            #     continue
            if (data['stimulation'].max() < stimulation_threshold):
                print(f"File {aligned_file} excluded due to stimulation being less than {stimulation_threshold}.")
                continue
            # Exclude the first and last 30 frames only if there are enough rows
            if len(data) > 60:
                data = data.iloc[30:-30]
            else:
                print(f"File {aligned_file} has fewer than 60 rows. Skipping frame exclusion.")
            
            # Remove rows with invalid X-Midpoint, Y-Midpoint, Ymin-bbox, or Ymax-bbox values
            data = data.dropna(subset=['X-Midpoint', 'Y-Midpoint', 'Ymin-bbox', 'Ymax-bbox'])
            data = data[(data['X-Midpoint'] > 0) & (data['Y-Midpoint'] > 0)]
            data = data[(data['Ymin-bbox'] != 0.0) & (data['Ymax-bbox'] != 0.0)]

            # Check for jumps in position
            x_midpoint = data['X-Midpoint'].values
            y_midpoint = data['Y-Midpoint'].values
            distances = np.sqrt(np.diff(x_midpoint)**2 + np.diff(y_midpoint)**2)
            if np.any(distances > 20):
                jump_index = np.where(distances > 20)[0][0] + 1  # Get the first jump index
                data = data.iloc[:jump_index]  # Exclude frames after the jump
            
            # Extract the midpoint coordinates
            x_midpoint = data['X-Midpoint']
            y_midpoint = data['Y-Midpoint']
            
            # Plot the path
            plt.plot(x_midpoint, y_midpoint, label=aligned_file, alpha=0.6)
    
    # Finalize the plot
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.title('All "Good" Aligned Larva Paths')
    plt.legend(loc='upper right', fontsize='small', bbox_to_anchor=(1.3, 1))
    plt.axis('equal')
    plt.grid()
    plt.show()

# # Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot'  # Replace with your base folder path
# plot_all_aligned_paths(base_folder)
# aligned_image_path1 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/aligned_PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
# aligned_image1 = pd.read_csv(aligned_image_path1).to_numpy()
# plt.imshow(aligned_image1, cmap='gray')
# plt.ylim(0,aligned_image1.shape[0])
# plt.title(f"Aligned Image:")
# plt.show()


## plot_all_aligned_paths

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_all_aligned_paths(base_folder):
    """
    Plot all aligned larva paths from a folder of folders on a single graph.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with aligned CSV files.

    Returns:
        None
    """
    # Initialize the plot
    plt.figure(figsize=(10, 10))
    
    # Loop through all subfolders in the base folder
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue
        
        # Find all aligned CSV files in the subfolder
        aligned_files = [f for f in os.listdir(folder_path) if f.startswith('aligned_2025') and f.endswith('.csv')]
        for aligned_file in aligned_files:
            aligned_file_path = os.path.join(folder_path, aligned_file)
            
            # Load the data
            data = pd.read_csv(aligned_file_path)
            
            # Check if required columns exist
            required_columns = ['X-Midpoint', 'Y-Midpoint', 'Ymin-bbox', 'Ymax-bbox']
            if not all(col in data.columns for col in required_columns):
                print(f"Required columns not found in {aligned_file}. Skipping.")
                continue
            
            # Check if "stimulation" is 0.0 for more than half the frames
            # if (data['stimulation'] == 0.0).sum() > len(data) / 4:
            #     print(f"File {aligned_file} excluded due to stimulation being 0.0 for more than half the frames.")
            #     continue
            # if (data['stimulation'].max() < 5000):
            #     print(f"File {aligned_file} excluded due to stimulation being less than 5000.")
            #     continue
            if (data['X-Midpoint'].max() > 5000):
                print(f"File {aligned_file} excluded due to X-Midpoint being more than 5000.")
                continue
            if (data['Y-Midpoint'].max() > 5000):
                print(f"File {aligned_file} excluded due to Y-Midpoint being more than 5000.")
                continue

            # Exclude the first and last 30 frames only if there are enough rows
            if len(data) > 60:
                data = data.iloc[30:-30]
            else:
                print(f"File {aligned_file} has fewer than 60 rows. Skipping frame exclusion.")
            
            # Remove rows with invalid X-Midpoint, Y-Midpoint, Ymin-bbox, or Ymax-bbox values
            data = data.dropna(subset=['X-Midpoint', 'Y-Midpoint', 'Ymin-bbox', 'Ymax-bbox'])
            data = data[(data['X-Midpoint'] > 0) & (data['Y-Midpoint'] > 0)]
            data = data[(data['Ymin-bbox'] != 0.0) & (data['Ymax-bbox'] != 0.0)]

            # Check for jumps in position
            x_midpoint = data['X-Midpoint'].values
            y_midpoint = data['Y-Midpoint'].values
            distances = np.sqrt(np.diff(x_midpoint)**2 + np.diff(y_midpoint)**2)
            if np.any(distances > 20):
                jump_index = np.where(distances > 20)[0][0] + 1  # Get the first jump index
                data = data.iloc[:jump_index]  # Exclude frames after the jump
            
            # Extract the midpoint coordinates
            x_midpoint = data['X-Midpoint']
            y_midpoint = data['Y-Midpoint']
            
            # Plot the path
            plt.plot(x_midpoint, y_midpoint, label=aligned_file, alpha=0.6)
    
    # Finalize the plot
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.title('All "Good" Aligned Larva Paths')
    plt.legend(loc='upper right', fontsize='small', bbox_to_anchor=(1.3, 1))
    plt.axis('equal')
    plt.grid()
    plt.show()

# # Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/average_plume'  # Replace with your base folder path
# plot_all_aligned_paths(base_folder)

# aligned_image_path1 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot/2025.05.05_18-31-52_PiVR4_snapshot_recenter/aligned_PANACHE1_crop_snapshot_1275_1.2[648,486,1.57].csv'
# aligned_image1 = pd.read_csv(aligned_image_path1).to_numpy()
# plt.imshow(aligned_image1, cmap='gray')
# plt.ylim(0,aligned_image1.shape[0])
# plt.title(f"Aligned Image:")
# plt.show()


## show image of arena

In [ ]:
def show_aligned_image_arena(base_folder):
    """
    Show the arena image from the first folder inside the base folder.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with the arena image.

    Returns:
        None
    """
    # Get the list of subfolders in the base folder
    subfolders = [f for f in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, f))]
    
    if not subfolders:
        print(f"No subfolders found in {base_folder}.")
        return
    
    # Use the first subfolder
    first_folder_path = os.path.join(base_folder, subfolders[0])
    
    # Find the aligned image file in the first subfolder
    arena_image_files = [f for f in os.listdir(first_folder_path) if f.startswith('aligned_') and f.endswith('].csv')]
    
    if not arena_image_files:
        print(f"No aligned arena image found in {first_folder_path}.")
        return
    
    # Load the first aligned image file
    arena_image_path = os.path.join(first_folder_path, arena_image_files[0])
    aligned_image = pd.read_csv(arena_image_path).to_numpy()
    
    # Display the image
    plt.imshow(aligned_image, cmap='gray')
    plt.ylim(0, aligned_image.shape[0])
    plt.title("Aligned Arena Image")
    plt.show()
    
    
    
    

# Use functions

In [ ]:
base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250501/snapshot' # Enter the path to your base folder
# Uncomment the following lines to run the functions

# align_all_images_in_folders(base_folder)
# show_aligned_images(base_folder)
# process_gui_corrected_files(base_folder)
# final_plot_all_good_aligned_paths(base_folder, 9900)  # Adjust the stimulation threshold as needed
# show_aligned_image_arena(base_folder)

base_folder2 = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot' # Enter the path to your base folder
# align_all_images_in_folders(base_folder2)
# show_aligned_images(base_folder2)
# process_gui_corrected_files(base_folder2)
# final_plot_all_good_aligned_paths(base_folder2, 9900)  # Adjust the stimulation threshold as needed
# show_aligned_image_arena(base_folder2)

base_folder3 = '/Users/eneko/Desktop/Louis Lab/data collection/250501/average_plume recenter' 
align_all_images_in_folders(base_folder3)
show_aligned_images(base_folder3)
# process_gui_corrected_files(base_folder3)
# final_plot_all_good_aligned_paths(base_folder3, 1000)  # Adjust the stimulation threshold as needed
# show_aligned_image_arena(base_folder3)

# New Method

## Determining rotation in PiVR

## Sanity check?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def calculate_angle(vector):
    """Calculate the angle (in degrees) between a vector and the horizontal axis."""
    x, y = vector
    angle = np.degrees(np.arctan2(y, x))  # atan2 handles the sign of x and y
    return angle

def find_starting_angle_moving_method(base_folder):
    """Process CSV files in multiple folders and plot angles on a dot chart."""
    folder_angles = {}

    # Loop through all subfolders in the base folder
    for folder_index, folder in enumerate(os.listdir(base_folder), start=1):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue

        folder_name = f"folder{folder_index}"
        folder_angles[folder_name] = []

        # Find all CSV files with names ending in "data_GUIcorrected.csv"
        csv_files = [f for f in os.listdir(folder_path) if f.endswith("data_GUIcorrected.csv") and not f.startswith("aligned_")]
        for csv_file in csv_files:
            csv_path = os.path.join(folder_path, csv_file)

            # Read the CSV file
            data = pd.read_csv(csv_path)

            # Ensure the required columns exist
            required_columns = ["Frame", "Time", "X-Centroid", "Y-Centroid"]
            if not all(col in data.columns for col in required_columns):
                print(f"Skipping {csv_file}: Missing required columns.")
                continue

            # Get the coordinates for the 1st and 120th frames
            if len(data) >= 120:
                x1, y1 = data.loc[0, ["X-Centroid", "Y-Centroid"]]
                x120, y120 = data.loc[119, ["X-Centroid", "Y-Centroid"]]
                

                # Calculate the vector and angle
                vector = (x120 - x1, y120 - y1)
                angle = calculate_angle(vector)
                folder_angles[folder_name].append(angle)
            else:
                print(f"Skipping {csv_file}: Not enough frames.")

    # Plot the angles on a dot chart
    plt.figure(figsize=(10, 6))
    for folder_name, angles in folder_angles.items():
        plt.scatter([folder_name] * len(angles), angles, label=folder_name, alpha=0.7)

    plt.xlabel("Folders")
    plt.ylabel("Angle (degrees)")
    plt.title("Dot Chart of Angles by Folder")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend(loc="upper right", fontsize="small")
    plt.show()

# Example usage
base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot recenter'  # Replace with your base folder path
process_csv_files(base_folder)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def calculate_angle(vector):
    """Calculate the angle (in degrees) between a vector and the horizontal axis."""
    x, y = vector
    angle = np.degrees(np.arctan2(y, x))  # atan2 handles the sign of x and y
    return angle

def find_starting_angle_tail_method(base_folder):
    """Find the starting angle using the average of angles made by the difference between centroid and tail coordinates."""
    folder_angles = {}

    # Loop through all subfolders in the base folder
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue

        folder_name = folder  # Use the actual folder name
        folder_angles[folder_name] = []

        # Find all CSV files with names ending in "data_GUIcorrected.csv"
        csv_files = [f for f in os.listdir(folder_path) if f.endswith("data_GUIcorrected.csv") and not f.startswith("aligned_")]
        for csv_file in csv_files:
            csv_path = os.path.join(folder_path, csv_file)

            # Read the CSV file
            data = pd.read_csv(csv_path)

            # Ensure the required columns exist
            required_columns = ["X-Centroid", "Y-Centroid", "X-Tail", "Y-Tail"]
            if not all(col in data.columns for col in required_columns):
                print(f"Skipping {csv_file}: Missing required columns.")
                continue

            # Check if there is tail data within the first 120 frames
            tail_data_start = data.iloc[:120][["X-Tail", "Y-Tail"]].dropna().index.min()
            if tail_data_start is None:
                print(f"Skipping {csv_file}: No tail data within the first 60 frames.")
                folder_angles[folder_name].append(1.0)
                continue

            # If tail data starts after the 120th frame, set angle to 0.0
            if tail_data_start >= 120:
                print(f"{csv_file}: Tail data starts after the 120th frame. Setting angle to 0.0.")
                folder_angles[folder_name].append(1.0)
                continue

            # Calculate angles from the start of tail data to the 120th frame
            angles = []
            for _, row in data.iloc[tail_data_start:120].iterrows():
                x_centroid, y_centroid = row["X-Centroid"], row["Y-Centroid"]
                x_tail, y_tail = row["X-Tail"], row["Y-Tail"]
                vector = (-(x_tail - x_centroid), -(y_tail - y_centroid))
                angle = calculate_angle(vector)
                angles.append(angle)

            # Calculate the average angle
            if angles:
                average_angle = np.mean(angles)
                folder_angles[folder_name].append(average_angle)

    # Plot the starting angles
    plt.figure(figsize=(10, 6))
    for folder_name, angles in folder_angles.items():
        if np.isnan(angles[0]):
            print(f"Skipping {folder_name} due to NaN angles.")
            continue
        plt.scatter([folder_name] * len(angles), angles, label=folder_name, alpha=0.7)

    plt.xlabel("Folders")
    plt.ylabel("Starting Angle (degrees)")
    plt.title("Starting Angles by Folder (Tail Method)")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend(loc="upper right", fontsize="small")
    plt.show()

    # Return the dictionary with folder names and angles
    return {folder: np.mean(angles) if angles else None for folder, angles in folder_angles.items()}

# Example usage
# base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot recenter'  # Replace with your base folder path
# angles_dict = find_starting_angle(base_folder)
# print(angles_dict)

In [ ]:
def plot_images_with_angles(base_folder, angles_dict):
    """Plot images with angles drawn on top."""
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue

        # Find the image file starting with "PANACHE1"
        image_files = [f for f in os.listdir(folder_path) if f.startswith("PANACHE1") and f.endswith(".csv")]
        if not image_files:
            print(f"No image file found in {folder_path}. Skipping.")
            continue

        image_file = image_files[0]
        image_path = os.path.join(folder_path, image_file)

        # Load the image as a NumPy array
        image = pd.read_csv(image_path, header=None).to_numpy()

        # Get the angle for the current folder
        angle = angles_dict.get(folder, None)
        if angle is None:
            print(f"No angle found for folder {folder}. Skipping.")
            continue

        # Plot the image
        plt.imshow(image, cmap="gray")

        # Draw the angle as a line in the middle of the image
        center_x, center_y = image.shape[1] // 2, image.shape[0] // 2
        length = 200  # Length of the line representing the angle
        end_x = center_x + length * np.cos(np.radians(angle))
        end_y = center_y - length * np.sin(np.radians(angle))  # Subtract because y-axis is inverted in images
        plt.arrow(center_x, center_y, end_x - center_x, end_y - center_y, color="red", head_width=20, head_length=30)

        plt.title(f"Folder: {folder}, Angle: {angle:.2f}°")
        plt.axis("off")
        plt.show()

# Example usage
base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot recenter'  # Replace with your base folder path
angles_dict = find_starting_angle_tail_method(base_folder)
plot_images_with_angles(base_folder, angles_dict)

# speed vs stimulation (average plume)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def plot_speed_vs_time(base_folder):
    """
    Read GUI-corrected files in the folders of the base folder, remove the first and last 60 frames,
    smooth the data, and plot the speed of the larvae with time.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with GUI-corrected files.

    Returns:
        None
    """
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue

        # Find the GUI-corrected file in the folder
        gui_corrected_files = [
            f for f in os.listdir(folder_path)
            if f.endswith("GUIcorrected.csv") and not f.startswith("aligned_")
        ]
        if not gui_corrected_files:
            print(f"No GUI-corrected file found in {folder_path}. Skipping.")
            continue

        gui_corrected_file = gui_corrected_files[0]
        gui_corrected_path = os.path.join(folder_path, gui_corrected_file)

        # Read the GUI-corrected file
        data = pd.read_csv(gui_corrected_path)

        # Ensure required columns exist
        if 'X-Centroid' not in data.columns or 'Y-Centroid' not in data.columns or 'Time' not in data.columns:
            print(f"Required columns not found in {gui_corrected_file}. Skipping.")
            continue

        # Remove the first and last 60 frames
        if len(data) > 120:
            data = data.iloc[60:-60]
        else:
            print(f"Not enough frames in {gui_corrected_file}. Skipping.")
            continue

        # Smooth the data using Savitzky-Golay filter
        window_length = 31  # Must be odd and less than the length of the data
        polyorder = 3       # Polynomial order for the filter
        data['X-Centroid'] = savgol_filter(data['X-Centroid'], window_length, polyorder)
        data['Y-Centroid'] = savgol_filter(data['Y-Centroid'], window_length, polyorder)

        # Calculate the speed
        x_centroid = data['X-Centroid'].values
        y_centroid = data['Y-Centroid'].values
        time = data['Time'].values

        # Calculate the differences in positions and time
        dx = np.diff(x_centroid)
        dy = np.diff(y_centroid)
        dt = np.diff(time)

        # Avoid division by zero
        dt[dt == 0] = np.nan

        # Calculate speed (pixels per unit time)
        speed = np.sqrt(dx**2 + dy**2) / dt

        # Plot speed vs time
        plt.figure(figsize=(10, 6))
        plt.plot(time[1:], speed, label=f"Folder: {folder}")
        plt.xlabel("Time (s)")
        plt.ylabel("Speed (pixels/s)")
        plt.title(f"Speed vs Time for {folder}")
        plt.grid(True)
        plt.legend()
        plt.show()

# Example usage
base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250506/snapshot recenter'  # Replace with your base folder path
plot_speed_vs_time(base_folder)

In [ ]:
from scipy.ndimage import label
def connected_components_1D(arr, dust_size=None):
    lbl, n_features = label(arr)
    indices = np.indices(lbl.shape).T[:, 0]
    cc_dict = {}
    for i in range(1, n_features + 1):
        cc_i = indices[lbl == i]
        if dust_size is not None:
            if np.shape(cc_i)[0] < dust_size:
                continue

        cc_dict[i] = cc_i

    return cc_dict

def cc_segments(array, dust_size=0):
    cc_dict = connected_components_1D(array, 
                                    dust_size=dust_size)
    return [cc_dict[x] for x in cc_dict.keys()]

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def plot_speed_vs_time_with_stimulation(base_folder, threshold=500):
    """
    Read GUI-corrected files in the folders of the base folder, remove the first and last 60 frames,
    smooth the data, and plot the speed of the larvae with time along with stimulation.

    Parameters:
        base_folder (str): Path to the base folder containing subfolders with GUI-corrected files.
        threshold (float): The stimulation threshold below which data is excluded.

    Returns:
        None
    """
    for folder in os.listdir(base_folder):
        folder_path = os.path.join(base_folder, folder)
        if not os.path.isdir(folder_path):
            continue

        # Find the GUI-corrected file in the folder
        gui_corrected_files = [
            f for f in os.listdir(folder_path)
            if f.endswith("GUIcorrected.csv") and not f.startswith("aligned_")
        ]
        if not gui_corrected_files:
            print(f"No GUI-corrected file found in {folder_path}. Skipping.")
            continue

        gui_corrected_file = gui_corrected_files[0]
        gui_corrected_path = os.path.join(folder_path, gui_corrected_file)

        # Read the GUI-corrected file
        data = pd.read_csv(gui_corrected_path)

        # Ensure required columns exist
        if 'X-Centroid' not in data.columns or 'Y-Centroid' not in data.columns or 'Time' not in data.columns or 'stimulation' not in data.columns:
            print(f"Required columns not found in {gui_corrected_file}. Skipping.")
            continue

        # Remove the first and last 60 frames
        if len(data) > 120:
            data = data.iloc[60:-60]
        else:
            print(f"Not enough frames in {gui_corrected_file}. Skipping.")
            continue

        # Smooth the data using Savitzky-Golay filter
        window_length = 31  # Must be odd and less than the length of the data
        polyorder = 3       # Polynomial order for the filter
        data['X-Centroid'] = savgol_filter(data['X-Centroid'], window_length, polyorder)
        data['Y-Centroid'] = savgol_filter(data['Y-Centroid'], window_length, polyorder)

        # Calculate the speed
        x_centroid = data['X-Centroid'].values
        y_centroid = data['Y-Centroid'].values
        time = data['Time'].values
        stimulation = data['stimulation'].values

        # Calculate the differences in positions and time
        dx = np.diff(x_centroid)
        dy = np.diff(y_centroid)
        dt = np.diff(time)

        # Avoid division by zero
        dt[dt == 0] = np.nan

        # Calculate speed (pixels per unit time)
        speed = np.sqrt(dx**2 + dy**2) / dt
        speed = np.clip(speed, 0, np.nanpercentile(speed, 95))  # Ensure speed is non-negative

        # Apply moving average to smooth the speed and stimulation data
        speed_smoothed = pd.Series(speed).rolling(window=window_length, center=True).mean().to_numpy()
        stimulation_smoothed = pd.Series(stimulation).rolling(window=window_length, center=True).mean().to_numpy()

        # Identify the last index where stimulation crosses the threshold
        below_threshold = stimulation_smoothed < threshold
        if np.any(below_threshold):
            last_below_index = np.argmax(below_threshold)  # First index where stimulation falls below the threshold
        else:
            last_below_index = len(stimulation_smoothed)  # Use the entire dataset if no threshold is crossed

        # Truncate the data
        time = time[:last_below_index]
        speed_smoothed = speed_smoothed[:last_below_index]
        stimulation_smoothed = stimulation_smoothed[:last_below_index]

        # Standardize speed and stimulation
        standardized_speed = (speed_smoothed - np.nanmean(speed_smoothed)) / np.nanstd(speed_smoothed)
        standardized_stimulation = (stimulation_smoothed - np.nanmean(stimulation_smoothed)) / np.nanstd(stimulation_smoothed)

        # Plot speed vs time with stimulation
        fig, ax1 = plt.subplots(figsize=(10, 6))

        # Plot standardized speed on the primary y-axis
        ax1.plot(time[1:], standardized_speed, label="Standardized Speed", color="blue")
        ax1.set_xlabel("Time (s)")
        ax1.set_ylabel("Standardized Speed", color="blue")
        ax1.tick_params(axis='y', labelcolor="blue")
        ax1.grid(True)

        # Plot standardized stimulation on the secondary y-axis
        ax2 = ax1.twinx()
        ax2.plot(time[:], standardized_stimulation, label="Standardized Stimulation", color="orange", alpha=0.7)
        ax2.set_ylabel("Standardized Stimulation", color="orange")
        ax2.tick_params(axis='y', labelcolor="orange")

        # Add title and legend
        plt.title(f"Standardized Speed and Stimulation vs Time for {folder}")
        fig.tight_layout()
        plt.show()

        # Calculate and print the correlation
        valid_indices = ~np.isnan(standardized_speed) & ~np.isnan(standardized_stimulation)
        correlation = np.corrcoef(standardized_speed[valid_indices], standardized_stimulation[valid_indices])[0, 1]
        print(f"Correlation for {folder}: {correlation}")


# Example usage
base_folder = '/Users/eneko/Desktop/Louis Lab/data collection/250421/gaussian'  # Replace with your base folder path
plot_speed_vs_time_with_stimulation(base_folder, threshold=500)